# 📊 Algorithmen — Das große Lern-Notebook

**Sortier-, Such- und Graphenalgorithmen in einem einzigen Notebook**

---
## 📚 Was du hier lernst
- **Sortieren:** Bubble Sort, Quick Sort, Merge Sort, Heap Sort
- **Suchen:** Lineare Suche, Binäre Suche
- **Graphen:** BFS, DFS, Dijkstra
- Laufzeitvergleich: O(n²) vs O(n log n) vs O(V+E)

> 💡 **Google Colab:** Datei → In Drive speichern → mit GPU/TPU ausführen.
> Alle Zellen von oben nach unten ausführen (Runtime → Run all).


In [ ]:
import time
import random
import heapq
import math
from collections import deque
from typing import TypeVar

import matplotlib.pyplot as plt

T = TypeVar("T")


# 🔄 Teil 1: Sortieralgorithmen

Sortieren ist die Grundlage vieler Algorithmen. Wir schauen uns vier klassische Verfahren an und vergleichen ihre Laufzeit.


## 🫧 Bubble Sort — O(n²)

Der einfachste Sortieralgorithmus: Vergleiche benachbarte Elemente und vertausche sie, wenn sie in der falschen Reihenfolge sind. Wiederhole, bis nichts mehr getauscht wird.


In [ ]:
def bubble_sort(arr: list[T]) -> list[T]:
    """O(n²) — einfach, aber langsam. Gut zum Lernen."""
    arr = arr.copy()
    n = len(arr)
    for i in range(n):
        swapped = False
        for j in range(n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        if not swapped:
            break
    return arr

# Test
arr = [64, 34, 25, 12, 22, 11, 90]
print(f"Original: {arr}")
print(f"Bubble:   {bubble_sort(arr)}")


## ⚡ Quick Sort — O(n log n)

**Divide & Conquer:** Wähle ein Pivot-Element, teile in "kleiner" und "größer", sortiere rekursiv.


In [ ]:
def quick_sort(arr: list[T]) -> list[T]:
    """O(n log n) average — Divide & Conquer."""
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)

print(f"Quick:    {quick_sort(arr)}")


## 🔀 Merge Sort — O(n log n)

Teile das Array in der Mitte, sortiere beide Hälften rekursiv, führe sie zusammen.


In [ ]:
def _merge(left: list[T], right: list[T]) -> list[T]:
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

def merge_sort(arr: list[T]) -> list[T]:
    """O(n log n) — stabil, gut für Linked Lists."""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return _merge(left, right)

print(f"Merge:    {merge_sort(arr)}")


## 🌲 Heap Sort — O(n log n)

**Idee:** Baue einen Max-Heap auf, entnimm wiederholt das größte Element und stelle die Heap-Eigenschaft wieder her.


In [ ]:
def _heapify(arr: list[T], n: int, i: int) -> None:
    largest = i
    left, right = 2 * i + 1, 2 * i + 2
    if left < n and arr[left] > arr[largest]:
        largest = left
    if right < n and arr[right] > arr[largest]:
        largest = right
    if largest != i:
        arr[i], arr[largest] = arr[largest], arr[i]
        _heapify(arr, n, largest)

def heap_sort(arr: list[T]) -> list[T]:
    """O(n log n) — in-place, nicht stabil."""
    arr = arr.copy()
    n = len(arr)
    for i in range(n // 2 - 1, -1, -1):
        _heapify(arr, n, i)
    for i in range(n - 1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]
        _heapify(arr, i, 0)
    return arr

print(f"Heap:     {heap_sort(arr)}")


## 📊 Laufzeitvergleich

Wie skalieren die Algorithmen mit der Array-Größe?


In [ ]:
sizes = [100, 500, 1000, 2000, 5000]
bubble_times = []
quick_times = []
merge_times = []
heap_times = []

for n in sizes:
    arr = [random.randint(0, 10000) for _ in range(n)]

    if n <= 2000:  # Bubble ist zu langsam für große n
        t0 = time.time()
        bubble_sort(arr)
        bubble_times.append(time.time() - t0)
    else:
        bubble_times.append(None)

    t0 = time.time()
    quick_sort(arr)
    quick_times.append(time.time() - t0)

    t0 = time.time()
    merge_sort(arr)
    merge_times.append(time.time() - t0)

    t0 = time.time()
    heap_sort(arr)
    heap_times.append(time.time() - t0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sizes[:len([t for t in bubble_times if t])], [t for t in bubble_times if t], 'o-', label='Bubble Sort O(n²)')
ax.plot(sizes, quick_times, 's-', label='Quick Sort O(n log n)')
ax.plot(sizes, merge_times, '^-', label='Merge Sort O(n log n)')
ax.plot(sizes, heap_times, 'd-', label='Heap Sort O(n log n)')
ax.set_xlabel('Array-Größe')
ax.set_ylabel('Zeit (Sekunden)')
ax.set_title('Sortieralgorithmen im Vergleich')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


# 🔍 Teil 2: Suchalgorithmen

Wie findet man ein Element in einer Datenstruktur? Zwei grundlegende Strategien.


## 🔎 Lineare Suche — O(n)

Gehe jedes Element der Reihe nach durch, bis du das Ziel findest. Funktioniert auf **unsortierten** Daten.


In [ ]:
def linear_search(arr: list[T], target: T) -> int:
    """O(n) — funktioniert auf unsortierten Daten."""
    for i, x in enumerate(arr):
        if x == target:
            return i
    return -1

beispiel = [64, 34, 25, 12, 22, 11, 90]
print(f"Linear Search 22: Index {linear_search(beispiel, 22)}")
print(f"Linear Search 99: Index {linear_search(beispiel, 99)} (nicht gefunden)")


## 🎯 Binary Search — O(log n)

In einem **sortierten** Array kannst du in log(n) Schritten finden, ob ein Element existiert.


In [ ]:
def binary_search(arr: list[T], target: T) -> int:
    """O(log n) — Voraussetzung: sortiertes Array."""
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

sorted_arr = [11, 12, 22, 25, 34, 64, 90]
print(f"Binary Search 22: Index {binary_search(sorted_arr, 22)}")
print(f"Binary Search 99: Index {binary_search(sorted_arr, 99)} (nicht gefunden)")

# Visualisierung
print(f"\nSuchraum bei 1.000.000 Elementen:")
print(f"  Linear: bis zu 1.000.000 Schritte")
print(f"  Binary: max {int(math.log2(1_000_000))} Schritte")


# 🕸️ Teil 3: Graphenalgorithmen

Graphen modellieren Beziehungen zwischen Objekten. Drei zentrale Traversierungs- und Pfad-Algorithmen.


## 🌊 BFS — Breitensuche

**Idee:** Erkunde Ebene für Ebene. Perfekt für kürzeste Pfade in ungewichteten Graphen.


In [ ]:
def bfs(graph: dict[T, list[T]], start: T) -> list[T]:
    """Breitensuche — kürzester Pfad in ungewichtetem Graphen."""
    visited = {start}
    queue = deque([start])
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order

# Beispiel-Graph
graph = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B"],
    "E": ["B", "F"],
    "F": ["C", "E"],
}

print(f"BFS ab A: {bfs(graph, 'A')}")
print(f"→ Ebene 0: A, Ebene 1: B,C, Ebene 2: D,E,F")


## 🔍 DFS — Tiefensuche

**Idee:** Gehe so tief wie möglich, dann zurück (Backtracking).


In [ ]:
def dfs(graph: dict[T, list[T]], start: T, visited: set | None = None) -> list[T]:
    """Tiefensuche — rekursiv."""
    if visited is None:
        visited = set()
    visited.add(start)
    order = [start]
    for neighbor in graph.get(start, []):
        if neighbor not in visited:
            order.extend(dfs(graph, neighbor, visited))
    return order

print(f"DFS ab A: {dfs(graph, 'A')}")
print(f"→ Geht erst tief (A→B→D), dann zurück")


## 🗺️ Dijkstra — Kürzeste Pfade mit Gewichten

**Idee:** Priority Queue — besuche immer den Knoten mit der geringsten Distanz zuerst.


In [ ]:
def dijkstra(graph: dict, start: T) -> dict:
    """Dijkstra — kürzeste Pfade in gewichtetem Graphen."""
    dist = {node: float("inf") for node in graph}
    dist[start] = 0
    pq = [(0, start)]
    while pq:
        d, node = heapq.heappop(pq)
        if d > dist[node]:
            continue
        for neighbor, weight in graph[node].items():
            new_dist = d + weight
            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                heapq.heappush(pq, (new_dist, neighbor))
    return dist

# Gewichteter Graph
weighted = {
    "A": {"B": 4, "C": 2},
    "B": {"A": 4, "C": 1, "D": 5},
    "C": {"A": 2, "B": 1, "D": 8, "E": 10},
    "D": {"B": 5, "C": 8, "E": 2},
    "E": {"C": 10, "D": 2},
}

dists = dijkstra(weighted, "A")
for node, d in sorted(dists.items()):
    print(f"A → {node}: {d}")

print(f"\nKürzester Pfad A→E: A→C→B→D→E = 2+1+5+2 = {dists['E']}")


## 📊 BFS vs DFS visuell

```
     A
    / \
   B   C
  / \ /
 D   E-F

BFS: A → B → C → D → E → F  (Ebene für Ebene)
DFS: A → B → D → E → F → C  (Tiefe zuerst)
```


# 🎯 Zusammenfassung

## Sortieren
| Algorithmus | Laufzeit | Stabil | In-Place |
|------------|----------|--------|----------|
| Bubble Sort | O(n²) | ✅ | ✅ |
| Quick Sort | O(n log n) | ❌ | ✅ |
| Merge Sort | O(n log n) | ✅ | ❌ |
| Heap Sort | O(n log n) | ❌ | ✅ |

## Suchen
| Algorithmus | Laufzeit | Voraussetzung |
|-------------|----------|---------------|
| Lineare Suche | O(n) | Keine |
| Binäre Suche | O(log n) | Sortiertes Array |

## Graphen
| Algorithmus | Laufzeit | Wofür? |
|------------|----------|--------|
| BFS | O(V+E) | Kürzester Pfad (ungewichtet) |
| DFS | O(V+E) | Zyklen finden, Topologische Sortierung |
| Dijkstra | O((V+E) log V) | Kürzester Pfad (gewichtet, positiv) |

> 📝 **Merke:** BFS mit Queue, DFS mit Stack/Rekursion, Dijkstra mit Priority Queue.
> Sortierte Daten sind der Schlüssel zu schnellen Suchoperationen!
